In [5]:
# ============================================================
# Validation Step 1 (Automated Rule-Based Validation)
# UPDATED SCOPE-BASED VERSION
#
# Key change:
# - Rates are now measured on rule-specific in-scope subsets,
#   not as ok / (ok + mismatch).
# - This makes meaningful missings count against the rate
#   whenever the rule should be available by definition.
#
# Scope policy:
# - Layer 1-related rules:
#     Overall -> full dataset
#     Base    -> Base_timing_regime == True
#
# - Layer 2 / step-telemetry-related rules:
#     Overall -> Step_telemetry == True
#     Base    -> Step_telemetry == True AND Base_timing_regime == True
#
# Reads from:
#   C:\Android Mobile App\ICST2026_Ext\MainDataset.csv
#   C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv
#     or C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.zip
#
# Writes to:
#   C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-1-Validation_Step_1\
# ============================================================

from pathlib import Path
import zipfile
import numpy as np
import pandas as pd

# ----------------------------
# Paths
# ----------------------------
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
IN_MAIN = BASE_DIR / "MainDataset.csv"
IN_STEPS_CSV = BASE_DIR / "run_steps_v16_stage3_breakdown.csv"
IN_STEPS_ZIP = BASE_DIR / "run_steps_v16_stage3_breakdown.zip"

OUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-1-Validation_Step_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-6

# ----------------------------
# Helpers
# ----------------------------
def to_dt(s):
    return pd.to_datetime(s, utc=True, errors="coerce")

def norm_bool_series(s):
    if s is None:
        return pd.Series(dtype="boolean")
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype("boolean")
    s = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": True, "false": False,
        "1": True, "0": False,
        "yes": True, "no": False,
        "y": True, "n": False,
        "t": True, "f": False,
    }
    return s.map(mapping).astype("boolean")

def yes_no_from_mask(mask):
    return np.where(mask, "Yes", "No")

def make_flag(applicable_mask, ok_mask):
    out = np.where(applicable_mask, np.where(ok_mask, "ok", "mismatch"), "missing")
    return pd.Series(out, index=applicable_mask.index if hasattr(applicable_mask, "index") else None)

def first_existing(df, cols, default=np.nan):
    for c in cols:
        if c in df.columns:
            return df[c]
    return pd.Series([default] * len(df), index=df.index)

def has_nonempty(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    return s != "" and s.lower() != "nan"

def present_flag(series):
    return series.map(has_nonempty)

def presence_rule(series, index):
    app = pd.Series(True, index=index)
    ok = present_flag(series)
    return make_flag(app, ok)

def canonicalize_priority_source(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if s == "" or s == "nan":
        return ""

    mapping = {
        "stage1_anchor_match": "stage1_anchor_match",
        "explicit_instru_execution_start": "explicit_instru_execution_start",
        "custom_followed_file_instru": "custom_followed_file_instru",
        "custom_stage1_supported_exec": "custom_stage1_supported_exec",
        "missing": "missing",
        "anchor_match": "stage1_anchor_match",
        "stage1_anchor": "stage1_anchor_match",
        "stage1_anchor_reasoned": "stage1_anchor_match",
        "explicit_instru": "explicit_instru_execution_start",
        "explicit_instrumentation": "explicit_instru_execution_start",
        "explicit_instru_start": "explicit_instru_execution_start",
        "explicit_execution_start": "explicit_instru_execution_start",
        "custom_supported": "custom_stage1_supported_exec",
    }
    return mapping.get(s, s)

# ----------------------------
# Read inputs
# ----------------------------
df = pd.read_csv(IN_MAIN, low_memory=False)

if IN_STEPS_CSV.exists():
    steps = pd.read_csv(IN_STEPS_CSV, low_memory=False)
elif IN_STEPS_ZIP.exists():
    with zipfile.ZipFile(IN_STEPS_ZIP) as zf:
        names = zf.namelist()
        if not names:
            raise FileNotFoundError("run_steps_v16_stage3_breakdown.zip is empty.")
        with zf.open(names[0]) as f:
            steps = pd.read_csv(f, low_memory=False)
else:
    raise FileNotFoundError("Could not find run_steps_v16_stage3_breakdown.csv or .zip")

# ----------------------------
# Normalize core columns
# ----------------------------
if "style" not in df.columns and "target_style" in df.columns:
    df["style"] = df["target_style"]

if "run_attempt" in df.columns and "attempt" not in df.columns:
    df["attempt"] = df["run_attempt"]

if "Base_timing_regime" in df.columns:
    base_timing_regime = norm_bool_series(df["Base_timing_regime"]).fillna(False)
else:
    controller_attempt_eq_1 = pd.to_numeric(first_existing(df, ["attempt", "run_attempt"]), errors="coerce").eq(1)
    rc = first_existing(df, ["run_conclusion", "conclusion"]).astype(str).str.lower()
    controller_usable_verdict = rc.isin(["success", "failure"])
    base_timing_regime = (controller_attempt_eq_1 & controller_usable_verdict).astype("boolean").fillna(False)

step_telemetry_bool = norm_bool_series(first_existing(df, ["Step_telemetry"])).fillna(False)
layer2_available_in_base_bool = norm_bool_series(first_existing(df, ["Layer2_available_in_base"])).fillna(False)

df["Base_timing_regime_bool"] = base_timing_regime
df["Step_telemetry_bool"] = step_telemetry_bool
df["Layer2_available_in_base_bool"] = layer2_available_in_base_bool

df["controlled_subset"] = yes_no_from_mask(df["Base_timing_regime_bool"])

# ----------------------------
# Parse datetime fields
# ----------------------------
run_start = to_dt(first_existing(df, ["study_run_boundary_start_at"]))
run_end = to_dt(first_existing(df, ["study_run_boundary_end_at"]))

inv_start = to_dt(first_existing(df, ["study_matched_invocation_step_started_at"]))
inv_end = to_dt(first_existing(df, ["study_matched_invocation_step_completed_at"]))
exec_end_start = to_dt(first_existing(df, ["study_invocation_execution_end_step_started_at"]))
exec_end_end = to_dt(first_existing(df, ["study_invocation_execution_end_step_completed_at"]))

window_start = to_dt(first_existing(df, ["study_invocation_execution_window_started_at"]))
window_end = to_dt(first_existing(df, ["study_invocation_execution_window_ended_at"]))

# ----------------------------
# Numeric fields
# ----------------------------
run_dur = pd.to_numeric(first_existing(df, ["study_run_duration_seconds"]), errors="coerce")

layer1_pre = pd.to_numeric(first_existing(df, ["study_layer1_time_to_instrumentation_envelope_seconds"]), errors="coerce")
layer1_mid = pd.to_numeric(first_existing(df, ["study_layer1_instrumentation_job_envelope_seconds"]), errors="coerce")
layer1_post = pd.to_numeric(first_existing(df, ["study_layer1_post_instrumentation_tail_seconds"]), errors="coerce")

l2_pre = pd.to_numeric(first_existing(df, ["study_pre_invocation_selected_stage3_seconds"]), errors="coerce")
l2_win = pd.to_numeric(first_existing(df, ["study_invocation_execution_window_selected_stage3_seconds"]), errors="coerce")
l2_post = pd.to_numeric(first_existing(df, ["study_post_invocation_selected_stage3_seconds"]), errors="coerce")

# ----------------------------
# Step-level source prep for V15-V18
# ----------------------------
steps["selected_invocation_cutpoint_bool"] = norm_bool_series(first_existing(steps, ["selected_invocation_cutpoint"])).fillna(False)
steps["selected_execution_end_cutpoint_bool"] = norm_bool_series(first_existing(steps, ["selected_execution_end_cutpoint"])).fillna(False)

for col in [
    "run_id", "job_ordinal_in_run", "step_ordinal_in_job",
    "step_name", "job_name", "started_at", "completed_at"
]:
    if col not in steps.columns:
        steps[col] = np.nan

steps["started_at_dt"] = to_dt(steps["started_at"])
steps["completed_at_dt"] = to_dt(steps["completed_at"])

inv_sel = (
    steps.loc[steps["selected_invocation_cutpoint_bool"]]
    .groupby("run_id", dropna=False)
    .agg(
        inv_selected_count=("selected_invocation_cutpoint_bool", "sum"),
        inv_step_name_source=("step_name", "first"),
        inv_job_name_source=("job_name", "first"),
        inv_job_ordinal_source=("job_ordinal_in_run", "first"),
        inv_step_ordinal_source=("step_ordinal_in_job", "first"),
        inv_started_at_source=("started_at_dt", "first"),
        inv_completed_at_source=("completed_at_dt", "first"),
    )
    .reset_index()
)

exe_sel = (
    steps.loc[steps["selected_execution_end_cutpoint_bool"]]
    .groupby("run_id", dropna=False)
    .agg(
        exe_selected_count=("selected_execution_end_cutpoint_bool", "sum"),
        exe_step_name_source=("step_name", "first"),
        exe_job_name_source=("job_name", "first"),
        exe_job_ordinal_source=("job_ordinal_in_run", "first"),
        exe_step_ordinal_source=("step_ordinal_in_job", "first"),
        exe_started_at_source=("started_at_dt", "first"),
        exe_completed_at_source=("completed_at_dt", "first"),
    )
    .reset_index()
)

df = df.merge(inv_sel, on="run_id", how="left")
df = df.merge(exe_sel, on="run_id", how="left")

stored_inv_step_name = first_existing(df, ["study_matched_invocation_step_name"])
stored_inv_job_name = first_existing(df, ["study_matched_invocation_job_name"])
stored_inv_job_ord = pd.to_numeric(first_existing(df, ["study_matched_invocation_job_ordinal_in_run"]), errors="coerce")
stored_inv_step_ord = pd.to_numeric(first_existing(df, ["study_matched_invocation_step_ordinal_in_job"]), errors="coerce")
stored_inv_started = to_dt(first_existing(df, ["study_matched_invocation_step_started_at"]))
stored_inv_completed = to_dt(first_existing(df, ["study_matched_invocation_step_completed_at"]))

stored_exe_step_name = first_existing(df, ["study_invocation_execution_end_step_name"])
stored_exe_job_name = first_existing(df, ["study_invocation_execution_end_job_name"])
stored_exe_job_ord = pd.to_numeric(first_existing(df, ["study_invocation_execution_end_job_ordinal_in_run"]), errors="coerce")
stored_exe_step_ord = pd.to_numeric(first_existing(df, ["study_invocation_execution_end_step_ordinal_in_job"]), errors="coerce")
stored_exe_started = to_dt(first_existing(df, ["study_invocation_execution_end_step_started_at"]))
stored_exe_completed = to_dt(first_existing(df, ["study_invocation_execution_end_step_completed_at"]))

# ----------------------------
# Rule flags
# ----------------------------

# V1
key_cols = [c for c in ["full_name", "run_id", "style"] if c in df.columns]
if len(key_cols) < 3:
    raise ValueError(f"Expected key columns full_name, run_id, style. Found: {key_cols}")
dup_mask = df.duplicated(subset=key_cols, keep=False)
df["v1_key_uniqueness_flag"] = np.where(dup_mask, "mismatch", "ok")

# V2
required_id_cols = [c for c in ["full_name", "run_id", "workflow_id", "workflow_identifier", "workflow_path"] if c in df.columns]
required_id_present = pd.concat([present_flag(df[c]) for c in required_id_cols], axis=1).all(axis=1)
df["v2_required_ids_flag"] = np.where(required_id_present, "ok", "mismatch")

# V3
v3_app = run_start.notna() & run_end.notna()
v3_ok = run_start <= run_end
df["v3_run_bounds_order_flag"] = make_flag(v3_app, v3_ok)

# V4
v4_app = window_start.notna() & window_end.notna()
v4_ok = window_start <= window_end
df["v4_window_bounds_order_flag"] = make_flag(v4_app, v4_ok)

# V5
v5_app = run_start.notna() & inv_start.notna() & exec_end_end.notna() & run_end.notna()
v5_ok = (run_start <= inv_start) & (inv_start <= exec_end_end) & (exec_end_end <= run_end)
df["v5_cutpoint_temporal_order_flag"] = make_flag(v5_app, v5_ok)

# V6
v6_app = run_start.notna() & window_start.notna() & window_end.notna() & run_end.notna()
v6_ok = (run_start <= window_start) & (window_start <= window_end) & (window_end <= run_end)
df["v6_window_inside_run_flag"] = make_flag(v6_app, v6_ok)

# V7
v7_app = run_dur.notna()
v7_ok = run_dur >= -EPS
df["v7_run_duration_nonnegative_flag"] = make_flag(v7_app, v7_ok)

# shared L2 applicability
layer2_value_app = l2_pre.notna() & l2_win.notna() & l2_post.notna() & run_dur.notna()

# V8
v8_ok = (l2_pre >= -EPS) & (l2_win >= -EPS) & (l2_post >= -EPS)
df["v8_layer2_nonnegative_flag"] = make_flag(layer2_value_app, v8_ok)

# V9
v9_ok = (l2_pre <= run_dur + EPS) & (l2_win <= run_dur + EPS) & (l2_post <= run_dur + EPS)
df["v9_layer2_bounded_by_run_flag"] = make_flag(layer2_value_app, v9_ok)

# V10
v10_app = layer1_pre.notna() & layer1_mid.notna() & layer1_post.notna() & run_dur.notna()
v10_ok = ((layer1_pre + layer1_mid + layer1_post) - run_dur).abs() <= EPS
df["v10_layer1_sum_to_run_flag"] = make_flag(v10_app, v10_ok)

# V11
v11_ok = ((l2_pre + l2_win + l2_post) - run_dur).abs() <= EPS
df["v11_window_decomposition_flag"] = make_flag(layer2_value_app, v11_ok)

# V12
v12_app = run_start.notna() & inv_start.notna() & l2_pre.notna()
v12_recomp = (inv_start - run_start).dt.total_seconds()
v12_ok = (v12_recomp - l2_pre).abs() <= EPS
df["v12_pre_invocation_recompute_flag"] = make_flag(v12_app, v12_ok)

# V13
v13_app = inv_start.notna() & exec_end_end.notna() & l2_win.notna()
v13_recomp = (exec_end_end - inv_start).dt.total_seconds()
v13_ok = (v13_recomp - l2_win).abs() <= EPS
df["v13_invocation_window_recompute_flag"] = make_flag(v13_app, v13_ok)

# V14
v14_app = exec_end_end.notna() & run_end.notna() & l2_post.notna()
v14_recomp = (run_end - exec_end_end).dt.total_seconds()
v14_ok = (v14_recomp - l2_post).abs() <= EPS
df["v14_post_invocation_recompute_flag"] = make_flag(v14_app, v14_ok)

# V15
v15_app = df["inv_selected_count"].notna()
v15_ok = df["inv_selected_count"].fillna(0).eq(1)
df["v15_unique_invocation_cutpoint_flag"] = make_flag(v15_app, v15_ok)

# V16
v16_app = df["exe_selected_count"].notna()
v16_ok = df["exe_selected_count"].fillna(0).eq(1)
df["v16_unique_execution_end_cutpoint_flag"] = make_flag(v16_app, v16_ok)

# V17
v17_app = (
    df["inv_selected_count"].fillna(0).eq(1)
    & stored_inv_step_name.map(has_nonempty)
    & stored_inv_job_name.map(has_nonempty)
    & stored_inv_started.notna()
)
v17_ok = (
    (stored_inv_step_name.astype(str).fillna("") == df["inv_step_name_source"].astype(str).fillna(""))
    & (stored_inv_job_name.astype(str).fillna("") == df["inv_job_name_source"].astype(str).fillna(""))
    & (stored_inv_job_ord.fillna(-999999) == pd.to_numeric(df["inv_job_ordinal_source"], errors="coerce").fillna(-999999))
    & (stored_inv_step_ord.fillna(-999999) == pd.to_numeric(df["inv_step_ordinal_source"], errors="coerce").fillna(-999999))
    & (stored_inv_started == to_dt(df["inv_started_at_source"]))
    & (stored_inv_completed == to_dt(df["inv_completed_at_source"]))
)
df["v17_invocation_step_source_match_flag"] = make_flag(v17_app, v17_ok)

# V18
v18_app = (
    df["exe_selected_count"].fillna(0).eq(1)
    & stored_exe_step_name.map(has_nonempty)
    & stored_exe_job_name.map(has_nonempty)
    & stored_exe_started.notna()
)
v18_ok = (
    (stored_exe_step_name.astype(str).fillna("") == df["exe_step_name_source"].astype(str).fillna(""))
    & (stored_exe_job_name.astype(str).fillna("") == df["exe_job_name_source"].astype(str).fillna(""))
    & (stored_exe_job_ord.fillna(-999999) == pd.to_numeric(df["exe_job_ordinal_source"], errors="coerce").fillna(-999999))
    & (stored_exe_step_ord.fillna(-999999) == pd.to_numeric(df["exe_step_ordinal_source"], errors="coerce").fillna(-999999))
    & (stored_exe_started == to_dt(df["exe_started_at_source"]))
    & (stored_exe_completed == to_dt(df["exe_completed_at_source"]))
)
df["v18_execution_end_step_source_match_flag"] = make_flag(v18_app, v18_ok)

# V19
allowed_styles = {"Community", "GMD", "Third-Party", "Custom"}
v19_app = df["style"].notna()
v19_ok = df["style"].astype(str).isin(allowed_styles)
df["v19_style_scope_valid_flag"] = make_flag(v19_app, v19_ok)

# Presence rules
df["v20_base_timing_regime_present"] = presence_rule(first_existing(df, ["Base_timing_regime"]), df.index)
df["v21_layer2_available_in_base_present"] = presence_rule(first_existing(df, ["Layer2_available_in_base"]), df.index)
df["v22_step_telemetry_present"] = presence_rule(first_existing(df, ["Step_telemetry"]), df.index)

df["v23_style_distinct_job_count_present"] = presence_rule(first_existing(df, ["study_style_distinct_job_count"]), df.index)
df["v24_style_distinct_job_base_count_present"] = presence_rule(first_existing(df, ["study_style_distinct_job_base_name_count"]), df.index)
df["v25_style_matrix_like_job_count_present"] = presence_rule(first_existing(df, ["study_style_matrix_like_job_count"]), df.index)
df["v26_style_matrix_expansion_flag_present"] = presence_rule(first_existing(df, ["study_style_matrix_expanded_flag"]), df.index)
df["v27_style_parallel_same_style_flag_present"] = presence_rule(first_existing(df, ["study_style_parallel_same_style_flag"]), df.index)
df["v28_style_max_parallel_jobs_present"] = presence_rule(first_existing(df, ["study_style_max_parallel_jobs"]), df.index)
df["v29_style_repeated_same_style_flag_present"] = presence_rule(first_existing(df, ["study_style_repeated_same_style_flag"]), df.index)

df["v30_invocation_candidate_total_count_present"] = presence_rule(first_existing(df, ["study_invocation_candidate_count_total"]), df.index)
df["v31_stage1_anchor_candidate_count_present"] = presence_rule(first_existing(df, ["study_stage1_anchor_candidate_count"]), df.index)
df["v32_explicit_instru_candidate_count_present"] = presence_rule(first_existing(df, ["study_explicit_instru_candidate_count"]), df.index)
df["v33_custom_supported_candidate_count_present"] = presence_rule(first_existing(df, ["study_custom_supported_candidate_count"]), df.index)
df["v34_distinct_invocation_candidate_step_name_count_present"] = presence_rule(first_existing(df, ["study_distinct_invocation_candidate_step_name_count"]), df.index)
df["v35_distinct_invocation_candidate_job_count_present"] = presence_rule(first_existing(df, ["study_distinct_invocation_candidate_job_count"]), df.index)
df["v36_selected_invocation_priority_source_present"] = presence_rule(first_existing(df, ["study_selected_invocation_priority_source"]), df.index)
df["v37_execution_window_candidate_count_present"] = presence_rule(first_existing(df, ["study_execution_window_candidate_count"]), df.index)
df["v38_execution_window_distinct_job_count_present"] = presence_rule(first_existing(df, ["study_execution_window_distinct_job_count"]), df.index)
df["v39_cross_job_execution_window_flag_present"] = presence_rule(first_existing(df, ["study_cross_job_execution_window_flag"]), df.index)

# Signature hash presence
study_signature_hash = first_existing(df, ["study_signature_hash"])
v40_app = step_telemetry_bool
v40_ok = present_flag(study_signature_hash)
df["v40_signature_hash_present"] = make_flag(v40_app, v40_ok)

df["v41_runner_os_bucket_present"] = presence_rule(first_existing(df, ["study_runner_os_bucket"]), df.index)
df["v42_job_count_exec_bucket_present"] = presence_rule(first_existing(df, ["study_job_count_exec_bucket"]), df.index)
df["v43_step_count_exec_bucket_present"] = presence_rule(first_existing(df, ["study_step_count_exec_bucket"]), df.index)

style_distinct_job_count = pd.to_numeric(first_existing(df, ["study_style_distinct_job_count"]), errors="coerce")
style_distinct_job_base_count = pd.to_numeric(first_existing(df, ["study_style_distinct_job_base_name_count"]), errors="coerce")
style_matrix_like_count = pd.to_numeric(first_existing(df, ["study_style_matrix_like_job_count"]), errors="coerce")
style_max_parallel_jobs = pd.to_numeric(first_existing(df, ["study_style_max_parallel_jobs"]), errors="coerce")

inv_total = pd.to_numeric(first_existing(df, ["study_invocation_candidate_count_total"]), errors="coerce")
inv_anchor_count = pd.to_numeric(first_existing(df, ["study_stage1_anchor_candidate_count"]), errors="coerce")
inv_explicit_count = pd.to_numeric(first_existing(df, ["study_explicit_instru_candidate_count"]), errors="coerce")
inv_custom_count = pd.to_numeric(first_existing(df, ["study_custom_supported_candidate_count"]), errors="coerce")
inv_distinct_step_name_count = pd.to_numeric(first_existing(df, ["study_distinct_invocation_candidate_step_name_count"]), errors="coerce")
inv_distinct_job_count = pd.to_numeric(first_existing(df, ["study_distinct_invocation_candidate_job_count"]), errors="coerce")
exec_window_candidate_count = pd.to_numeric(first_existing(df, ["study_execution_window_candidate_count"]), errors="coerce")
exec_window_distinct_job_count = pd.to_numeric(first_existing(df, ["study_execution_window_distinct_job_count"]), errors="coerce")

# V44
v44_app = style_distinct_job_base_count.notna() & style_distinct_job_count.notna()
v44_ok = style_distinct_job_base_count <= style_distinct_job_count
df["v44_job_base_count_bounded_flag"] = make_flag(v44_app, v44_ok)

# V45
v45_app = style_matrix_like_count.notna() & style_distinct_job_count.notna()
v45_ok = style_matrix_like_count <= style_distinct_job_count
df["v45_matrix_like_count_bounded_flag"] = make_flag(v45_app, v45_ok)

# V46
v46_app = style_max_parallel_jobs.notna() & style_distinct_job_count.notna()
v46_ok = style_max_parallel_jobs <= style_distinct_job_count
df["v46_max_parallel_jobs_bounded_flag"] = make_flag(v46_app, v46_ok)

# V47
v47_app = inv_distinct_step_name_count.notna() & inv_total.notna()
v47_ok = inv_distinct_step_name_count <= inv_total
df["v47_distinct_invocation_step_name_count_bounded_flag"] = make_flag(v47_app, v47_ok)

# V48
v48_app = inv_distinct_job_count.notna() & inv_total.notna()
v48_ok = inv_distinct_job_count <= inv_total
df["v48_distinct_invocation_job_count_bounded_flag"] = make_flag(v48_app, v48_ok)

# V49
v49_app = inv_total.notna() & inv_anchor_count.notna() & inv_explicit_count.notna() & inv_custom_count.notna()
v49_ok = (inv_anchor_count + inv_explicit_count + inv_custom_count) <= inv_total
df["v49_invocation_candidate_partition_flag"] = make_flag(v49_app, v49_ok)

# V50
v50_app = exec_window_distinct_job_count.notna() & exec_window_candidate_count.notna()
v50_ok = exec_window_distinct_job_count <= exec_window_candidate_count
df["v50_execution_window_distinct_job_bounded_flag"] = make_flag(v50_app, v50_ok)

# V51
style_matrix_expanded = norm_bool_series(first_existing(df, ["study_style_matrix_expanded_flag"])).fillna(False)
style_repeated_same_style = norm_bool_series(first_existing(df, ["study_style_repeated_same_style_flag"])).fillna(False)
v51_app = present_flag(first_existing(df, ["study_style_matrix_expanded_flag"])) & present_flag(first_existing(df, ["study_style_repeated_same_style_flag"]))
v51_ok = (~style_matrix_expanded) | style_repeated_same_style
df["v51_matrix_expanded_implies_repeated_flag"] = make_flag(v51_app, v51_ok)

# V52
cross_job_exec_window = norm_bool_series(first_existing(df, ["study_cross_job_execution_window_flag"])).fillna(False)
v52_app = present_flag(first_existing(df, ["study_cross_job_execution_window_flag"])) & exec_window_distinct_job_count.notna()
v52_ok = ((~cross_job_exec_window) & exec_window_distinct_job_count.fillna(0).le(1)) | (
    cross_job_exec_window & exec_window_distinct_job_count.fillna(0).ge(2)
)
df["v52_cross_job_window_flag_consistency"] = make_flag(v52_app, v52_ok)

# V53
style_parallel_same_style = norm_bool_series(first_existing(df, ["study_style_parallel_same_style_flag"])).fillna(False)
v53_app = present_flag(first_existing(df, ["study_style_parallel_same_style_flag"])) & style_max_parallel_jobs.notna()
v53_ok = ((~style_parallel_same_style) & style_max_parallel_jobs.fillna(0).le(1)) | (
    style_parallel_same_style & style_max_parallel_jobs.fillna(0).ge(2)
)
df["v53_parallel_same_style_flag_consistency"] = make_flag(v53_app, v53_ok)

# V54
priority_source_raw = first_existing(df, ["study_selected_invocation_priority_source"])
priority_source_canon = priority_source_raw.map(canonicalize_priority_source)
matched_invocation_started_at = to_dt(first_existing(df, ["study_matched_invocation_step_started_at"]))

valid_priority_sources = {
    "stage1_anchor_match",
    "explicit_instru_execution_start",
    "custom_followed_file_instru",
    "custom_stage1_supported_exec",
    "missing",
}

v54_app = priority_source_canon.map(has_nonempty)
v54_label_valid = priority_source_canon.isin(valid_priority_sources)

v54_context_ok = (
    ((priority_source_canon == "missing") & matched_invocation_started_at.isna()) |
    ((priority_source_canon == "stage1_anchor_match")
        & matched_invocation_started_at.notna()
        & inv_anchor_count.fillna(0).ge(1)
        & inv_total.fillna(0).ge(1)) |
    ((priority_source_canon == "explicit_instru_execution_start")
        & matched_invocation_started_at.notna()
        & inv_explicit_count.fillna(0).ge(1)
        & inv_total.fillna(0).ge(1)) |
    ((priority_source_canon == "custom_followed_file_instru")
        & matched_invocation_started_at.notna()
        & inv_total.fillna(0).ge(1)) |
    ((priority_source_canon == "custom_stage1_supported_exec")
        & matched_invocation_started_at.notna()
        & inv_custom_count.fillna(0).ge(1)
        & inv_total.fillna(0).ge(1))
)

v54_ok = v54_label_valid & v54_context_ok
df["v54_selected_invocation_priority_source_validity"] = make_flag(v54_app, v54_ok)

# ----------------------------
# Rule list
# ----------------------------
flag_cols = [
    "v1_key_uniqueness_flag",
    "v2_required_ids_flag",
    "v3_run_bounds_order_flag",
    "v4_window_bounds_order_flag",
    "v5_cutpoint_temporal_order_flag",
    "v6_window_inside_run_flag",
    "v7_run_duration_nonnegative_flag",
    "v8_layer2_nonnegative_flag",
    "v9_layer2_bounded_by_run_flag",
    "v10_layer1_sum_to_run_flag",
    "v11_window_decomposition_flag",
    "v12_pre_invocation_recompute_flag",
    "v13_invocation_window_recompute_flag",
    "v14_post_invocation_recompute_flag",
    "v15_unique_invocation_cutpoint_flag",
    "v16_unique_execution_end_cutpoint_flag",
    "v17_invocation_step_source_match_flag",
    "v18_execution_end_step_source_match_flag",
    "v19_style_scope_valid_flag",
    "v20_base_timing_regime_present",
    "v21_layer2_available_in_base_present",
    "v22_step_telemetry_present",
    "v23_style_distinct_job_count_present",
    "v24_style_distinct_job_base_count_present",
    "v25_style_matrix_like_job_count_present",
    "v26_style_matrix_expansion_flag_present",
    "v27_style_parallel_same_style_flag_present",
    "v28_style_max_parallel_jobs_present",
    "v29_style_repeated_same_style_flag_present",
    "v30_invocation_candidate_total_count_present",
    "v31_stage1_anchor_candidate_count_present",
    "v32_explicit_instru_candidate_count_present",
    "v33_custom_supported_candidate_count_present",
    "v34_distinct_invocation_candidate_step_name_count_present",
    "v35_distinct_invocation_candidate_job_count_present",
    "v36_selected_invocation_priority_source_present",
    "v37_execution_window_candidate_count_present",
    "v38_execution_window_distinct_job_count_present",
    "v39_cross_job_execution_window_flag_present",
    "v40_signature_hash_present",
    "v41_runner_os_bucket_present",
    "v42_job_count_exec_bucket_present",
    "v43_step_count_exec_bucket_present",
    "v44_job_base_count_bounded_flag",
    "v45_matrix_like_count_bounded_flag",
    "v46_max_parallel_jobs_bounded_flag",
    "v47_distinct_invocation_step_name_count_bounded_flag",
    "v48_distinct_invocation_job_count_bounded_flag",
    "v49_invocation_candidate_partition_flag",
    "v50_execution_window_distinct_job_bounded_flag",
    "v51_matrix_expanded_implies_repeated_flag",
    "v52_cross_job_window_flag_consistency",
    "v53_parallel_same_style_flag_consistency",
    "v54_selected_invocation_priority_source_validity",
]

# ----------------------------
# Rule scope map
# ----------------------------
# scope_group:
# - "layer1_full_or_base" => Overall: full dataset | Base: Base_timing_regime
# - "layer2_steptelemetry" => Overall: Step_telemetry | Base: Step_telemetry & Base_timing_regime
# - "meta_full_or_base"    => same as Layer 1 for broad study-facing/meta fields

rule_scope_group = {
    "v1_key_uniqueness_flag": "meta_full_or_base",
    "v2_required_ids_flag": "meta_full_or_base",
    "v3_run_bounds_order_flag": "layer1_full_or_base",
    "v4_window_bounds_order_flag": "layer2_steptelemetry",
    "v5_cutpoint_temporal_order_flag": "layer2_steptelemetry",
    "v6_window_inside_run_flag": "layer2_steptelemetry",
    "v7_run_duration_nonnegative_flag": "layer1_full_or_base",
    "v8_layer2_nonnegative_flag": "layer2_steptelemetry",
    "v9_layer2_bounded_by_run_flag": "layer2_steptelemetry",
    "v10_layer1_sum_to_run_flag": "layer1_full_or_base",
    "v11_window_decomposition_flag": "layer2_steptelemetry",
    "v12_pre_invocation_recompute_flag": "layer2_steptelemetry",
    "v13_invocation_window_recompute_flag": "layer2_steptelemetry",
    "v14_post_invocation_recompute_flag": "layer2_steptelemetry",
    "v15_unique_invocation_cutpoint_flag": "layer2_steptelemetry",
    "v16_unique_execution_end_cutpoint_flag": "layer2_steptelemetry",
    "v17_invocation_step_source_match_flag": "layer2_steptelemetry",
    "v18_execution_end_step_source_match_flag": "layer2_steptelemetry",
    "v19_style_scope_valid_flag": "meta_full_or_base",
    "v20_base_timing_regime_present": "meta_full_or_base",
    "v21_layer2_available_in_base_present": "meta_full_or_base",
    "v22_step_telemetry_present": "meta_full_or_base",
    "v23_style_distinct_job_count_present": "meta_full_or_base",
    "v24_style_distinct_job_base_count_present": "meta_full_or_base",
    "v25_style_matrix_like_job_count_present": "meta_full_or_base",
    "v26_style_matrix_expansion_flag_present": "meta_full_or_base",
    "v27_style_parallel_same_style_flag_present": "meta_full_or_base",
    "v28_style_max_parallel_jobs_present": "meta_full_or_base",
    "v29_style_repeated_same_style_flag_present": "meta_full_or_base",
    "v30_invocation_candidate_total_count_present": "meta_full_or_base",
    "v31_stage1_anchor_candidate_count_present": "meta_full_or_base",
    "v32_explicit_instru_candidate_count_present": "meta_full_or_base",
    "v33_custom_supported_candidate_count_present": "meta_full_or_base",
    "v34_distinct_invocation_candidate_step_name_count_present": "meta_full_or_base",
    "v35_distinct_invocation_candidate_job_count_present": "meta_full_or_base",
    "v36_selected_invocation_priority_source_present": "meta_full_or_base",
    "v37_execution_window_candidate_count_present": "layer2_steptelemetry",
    "v38_execution_window_distinct_job_count_present": "layer2_steptelemetry",
    "v39_cross_job_execution_window_flag_present": "layer2_steptelemetry",
    "v40_signature_hash_present": "layer2_steptelemetry",
    "v41_runner_os_bucket_present": "meta_full_or_base",
    "v42_job_count_exec_bucket_present": "meta_full_or_base",
    "v43_step_count_exec_bucket_present": "meta_full_or_base",
    "v44_job_base_count_bounded_flag": "meta_full_or_base",
    "v45_matrix_like_count_bounded_flag": "meta_full_or_base",
    "v46_max_parallel_jobs_bounded_flag": "meta_full_or_base",
    "v47_distinct_invocation_step_name_count_bounded_flag": "meta_full_or_base",
    "v48_distinct_invocation_job_count_bounded_flag": "meta_full_or_base",
    "v49_invocation_candidate_partition_flag": "meta_full_or_base",
    "v50_execution_window_distinct_job_bounded_flag": "layer2_steptelemetry",
    "v51_matrix_expanded_implies_repeated_flag": "meta_full_or_base",
    "v52_cross_job_window_flag_consistency": "layer2_steptelemetry",
    "v53_parallel_same_style_flag_consistency": "meta_full_or_base",
    "v54_selected_invocation_priority_source_validity": "meta_full_or_base",
}

# ----------------------------
# Scope-aware summaries
# ----------------------------
def get_scope_mask(df_in, scope_group, view):
    if scope_group in {"layer1_full_or_base", "meta_full_or_base"}:
        if view == "Overall":
            return pd.Series(True, index=df_in.index)
        if view == "Base":
            return df_in["Base_timing_regime_bool"].fillna(False)
    elif scope_group == "layer2_steptelemetry":
        if view == "Overall":
            return df_in["Step_telemetry_bool"].fillna(False)
        if view == "Base":
            return df_in["Step_telemetry_bool"].fillna(False) & df_in["Base_timing_regime_bool"].fillna(False)
    raise ValueError(f"Unknown scope_group/view combination: {scope_group} / {view}")

def summarize_flags_scope_based(df_in, flag_cols):
    rows = []
    for view in ["Overall", "Base"]:
        for c in flag_cols:
            scope_group = rule_scope_group[c]
            scope_mask = get_scope_mask(df_in, scope_group, view)
            sub = df_in.loc[scope_mask, c]

            rows_in_scope = int(len(sub))
            ok_count = int((sub == "ok").sum())
            mismatch_count = int((sub == "mismatch").sum())
            missing_count = int((sub == "missing").sum())

            ok_rate_pct = np.nan if rows_in_scope == 0 else round(ok_count * 100.0 / rows_in_scope, 4)
            mismatch_rate_pct = np.nan if rows_in_scope == 0 else round(mismatch_count * 100.0 / rows_in_scope, 4)
            missing_rate_pct = np.nan if rows_in_scope == 0 else round(missing_count * 100.0 / rows_in_scope, 4)

            rows.append({
                "view": view,
                "check_name": c,
                "scope_group": scope_group,
                "rows_in_scope": rows_in_scope,
                "ok_count": ok_count,
                "mismatch_count": mismatch_count,
                "missing_count": missing_count,
                "ok_rate_pct": ok_rate_pct,
                "mismatch_rate_pct": mismatch_rate_pct,
                "missing_rate_pct": missing_rate_pct,
            })
    return pd.DataFrame(rows)

summary_scope_based = summarize_flags_scope_based(df, flag_cols)
summary_scope_based.to_csv(OUT_DIR / "validation_step1_scope_based_summary.csv", index=False)

# ----------------------------
# Category-level summaries
# ----------------------------
category_map = {
    "Structural integrity": [
        "v1_key_uniqueness_flag",
        "v2_required_ids_flag",
        "v19_style_scope_valid_flag",
        "v20_base_timing_regime_present",
        "v21_layer2_available_in_base_present",
        "v22_step_telemetry_present",
    ],
    "Temporal consistency": [
        "v3_run_bounds_order_flag",
        "v4_window_bounds_order_flag",
        "v5_cutpoint_temporal_order_flag",
        "v6_window_inside_run_flag",
    ],
    "Layer 1 equation validity": [
        "v10_layer1_sum_to_run_flag",
    ],
    "Layer 2 recomputation validity": [
        "v8_layer2_nonnegative_flag",
        "v9_layer2_bounded_by_run_flag",
        "v11_window_decomposition_flag",
        "v12_pre_invocation_recompute_flag",
        "v13_invocation_window_recompute_flag",
        "v14_post_invocation_recompute_flag",
    ],
    "Source traceability": [
        "v15_unique_invocation_cutpoint_flag",
        "v16_unique_execution_end_cutpoint_flag",
        "v17_invocation_step_source_match_flag",
        "v18_execution_end_step_source_match_flag",
    ],
    "Study-facing field integrity": [
        "v40_signature_hash_present",
        "v41_runner_os_bucket_present",
        "v42_job_count_exec_bucket_present",
        "v43_step_count_exec_bucket_present",
    ],
    "Auxiliary consistency": [
        "v23_style_distinct_job_count_present",
        "v24_style_distinct_job_base_count_present",
        "v25_style_matrix_like_job_count_present",
        "v26_style_matrix_expansion_flag_present",
        "v27_style_parallel_same_style_flag_present",
        "v28_style_max_parallel_jobs_present",
        "v29_style_repeated_same_style_flag_present",
        "v30_invocation_candidate_total_count_present",
        "v31_stage1_anchor_candidate_count_present",
        "v32_explicit_instru_candidate_count_present",
        "v33_custom_supported_candidate_count_present",
        "v34_distinct_invocation_candidate_step_name_count_present",
        "v35_distinct_invocation_candidate_job_count_present",
        "v36_selected_invocation_priority_source_present",
        "v37_execution_window_candidate_count_present",
        "v38_execution_window_distinct_job_count_present",
        "v39_cross_job_execution_window_flag_present",
        "v44_job_base_count_bounded_flag",
        "v45_matrix_like_count_bounded_flag",
        "v46_max_parallel_jobs_bounded_flag",
        "v47_distinct_invocation_step_name_count_bounded_flag",
        "v48_distinct_invocation_job_count_bounded_flag",
        "v49_invocation_candidate_partition_flag",
        "v50_execution_window_distinct_job_bounded_flag",
        "v51_matrix_expanded_implies_repeated_flag",
        "v52_cross_job_window_flag_consistency",
        "v53_parallel_same_style_flag_consistency",
        "v54_selected_invocation_priority_source_validity",
    ],
}

def category_summary_scope_based(df_in):
    rows = []
    for view in ["Overall", "Base"]:
        for cat, rules in category_map.items():
            vals = []
            total_scope = 0
            total_ok = 0
            total_mismatch = 0
            total_missing = 0

            for r in rules:
                scope_group = rule_scope_group[r]
                scope_mask = get_scope_mask(df_in, scope_group, view)
                sub = df_in.loc[scope_mask, r]

                rows_in_scope = int(len(sub))
                ok_count = int((sub == "ok").sum())
                mismatch_count = int((sub == "mismatch").sum())
                missing_count = int((sub == "missing").sum())

                total_scope += rows_in_scope
                total_ok += ok_count
                total_mismatch += mismatch_count
                total_missing += missing_count

                if rows_in_scope > 0:
                    vals.append(ok_count * 100.0 / rows_in_scope)

            rows.append({
                "view": view,
                "category": cat,
                "avg_scope_based_ok_rate_pct": round(float(np.mean(vals)), 2) if vals else np.nan,
                "rows_in_scope_total_across_rules": total_scope,
                "ok_count_total_across_rules": total_ok,
                "mismatch_count_total_across_rules": total_mismatch,
                "missing_count_total_across_rules": total_missing,
            })
    return pd.DataFrame(rows)

category_scope_based = category_summary_scope_based(df)
category_scope_based.to_csv(OUT_DIR / "validation_step1_scope_based_category_summary.csv", index=False)

# ----------------------------
# Record-level flags
# ----------------------------
record_cols = key_cols + ["workflow_id", "workflow_identifier", "workflow_path", "run_attempt", "attempt", "style", "controlled_subset"]
record_cols = [c for c in record_cols if c in df.columns]
record_flags = df[record_cols + ["Base_timing_regime_bool", "Step_telemetry_bool"] + flag_cols].copy()
record_flags.to_csv(OUT_DIR / "validation_step1_record_flags.csv", index=False)

# ----------------------------
# Scope-based issue outputs
# ----------------------------
def scoped_issue_extract(df_in, rules, view_name, fname):
    issue_mask = pd.Series(False, index=df_in.index)
    for r in rules:
        scope_mask = get_scope_mask(df_in, rule_scope_group[r], view_name)
        issue_mask |= scope_mask & df_in[r].isin(["mismatch", "missing"])
    out = df_in.loc[issue_mask, record_cols + ["Base_timing_regime_bool", "Step_telemetry_bool"] + rules].copy()
    out.to_csv(OUT_DIR / fname, index=False)

cutpoint_issue_rules = [
    "v5_cutpoint_temporal_order_flag",
    "v6_window_inside_run_flag",
    "v11_window_decomposition_flag",
    "v12_pre_invocation_recompute_flag",
    "v13_invocation_window_recompute_flag",
    "v14_post_invocation_recompute_flag",
    "v15_unique_invocation_cutpoint_flag",
    "v16_unique_execution_end_cutpoint_flag",
]
scoped_issue_extract(df, cutpoint_issue_rules, "Overall", "validation_step1_cutpoint_issues_scope_overall.csv")
scoped_issue_extract(df, cutpoint_issue_rules, "Base", "validation_step1_cutpoint_issues_scope_base.csv")

step_match_rules = [
    "v15_unique_invocation_cutpoint_flag",
    "v16_unique_execution_end_cutpoint_flag",
    "v17_invocation_step_source_match_flag",
    "v18_execution_end_step_source_match_flag",
]
scoped_issue_extract(df, step_match_rules, "Overall", "validation_step1_step_match_issues_scope_overall.csv")
scoped_issue_extract(df, step_match_rules, "Base", "validation_step1_step_match_issues_scope_base.csv")

aux_rules = [
    "v20_base_timing_regime_present",
    "v21_layer2_available_in_base_present",
    "v22_step_telemetry_present",
    "v23_style_distinct_job_count_present",
    "v24_style_distinct_job_base_count_present",
    "v25_style_matrix_like_job_count_present",
    "v26_style_matrix_expansion_flag_present",
    "v27_style_parallel_same_style_flag_present",
    "v28_style_max_parallel_jobs_present",
    "v29_style_repeated_same_style_flag_present",
    "v30_invocation_candidate_total_count_present",
    "v31_stage1_anchor_candidate_count_present",
    "v32_explicit_instru_candidate_count_present",
    "v33_custom_supported_candidate_count_present",
    "v34_distinct_invocation_candidate_step_name_count_present",
    "v35_distinct_invocation_candidate_job_count_present",
    "v36_selected_invocation_priority_source_present",
    "v37_execution_window_candidate_count_present",
    "v38_execution_window_distinct_job_count_present",
    "v39_cross_job_execution_window_flag_present",
    "v40_signature_hash_present",
    "v41_runner_os_bucket_present",
    "v42_job_count_exec_bucket_present",
    "v43_step_count_exec_bucket_present",
    "v44_job_base_count_bounded_flag",
    "v45_matrix_like_count_bounded_flag",
    "v46_max_parallel_jobs_bounded_flag",
    "v47_distinct_invocation_step_name_count_bounded_flag",
    "v48_distinct_invocation_job_count_bounded_flag",
    "v49_invocation_candidate_partition_flag",
    "v50_execution_window_distinct_job_bounded_flag",
    "v51_matrix_expanded_implies_repeated_flag",
    "v52_cross_job_window_flag_consistency",
    "v53_parallel_same_style_flag_consistency",
    "v54_selected_invocation_priority_source_validity",
]
scoped_issue_extract(df, aux_rules, "Overall", "validation_step1_auxiliary_issues_scope_overall.csv")
scoped_issue_extract(df, aux_rules, "Base", "validation_step1_auxiliary_issues_scope_base.csv")

# ----------------------------
# Console output
# ----------------------------
print("\n=== Scope-based category summary ===")
print(category_scope_based.to_string(index=False))

print("\n=== Saved files to ===")
print(OUT_DIR)


=== Scope-based category summary ===
   view                       category  avg_scope_based_ok_rate_pct  rows_in_scope_total_across_rules  ok_count_total_across_rules  mismatch_count_total_across_rules  missing_count_total_across_rules
Overall           Structural integrity                       100.00                             53070                        53070                                  0                                 0
Overall           Temporal consistency                       100.00                             26086                        26086                                  0                                 0
Overall      Layer 1 equation validity                        98.45                              8845                         8708                                  0                               137
Overall Layer 2 recomputation validity                       100.00                             34482                        34482                                